this here is the data pipeline for turning the data into tuples!

In [15]:
#load packages
import numpy as np
import mne
import pandas as pd
from scipy.signal import coherence
from scipy.signal import welch

In [ ]:
# loading data from:https://physionet.org/content/eegmat/1.0.0/
# dataset has a low-pass filter at 30Hz so gamma (30+Hz) has minimal signal
# _1 files = baseline EEG, _2 files = EEG during mental arithmetic task
# load files in subject order, trim each one before combining
files = [
    ('Subject00_1.edf', 0, 'baseline'),
    ('Subject00_2.edf', 0, 'task'),
    ('Subject01_1.edf', 1, 'baseline'),
    ('Subject01_2.edf', 1, 'task'),
    ('Subject02_1.edf', 2, 'baseline'),
    ('Subject02_2.edf', 2, 'task'),
    ('Subject03_1.edf', 3, 'baseline'),
    ('Subject03_2.edf', 3, 'task'),
    ('Subject04_1.edf', 4, 'baseline'),
    ('Subject04_2.edf', 4, 'task'),
    ('Subject05_1.edf', 5, 'baseline'),
    ('Subject05_2.edf', 5, 'task'),
]
trimmed = []

for f, subject_id, session in files:
    raw = mne.io.read_raw_edf(f, preload=True, encoding='latin1')
    temp = raw.to_data_frame()
    temp = temp.drop(columns=['ECG ECG'])
    
    # trim flat padding off the end of each file
    eeg_data = temp[[col for col in temp.columns if col.startswith('EEG')]]
    changing = (eeg_data.diff() != 0).any(axis=1)
    last_valid = changing[::-1].idxmax()
    temp = temp.iloc[:last_valid + 1]
    
    # tag each row with subject and session so teammates know the source
    temp['subject_id'] = subject_id
    temp['session'] = session
    
    print(f'{f}: {len(temp)} valid rows')
    trimmed.append(temp)

df = pd.concat(trimmed, ignore_index=True)
df

Extracting EDF parameters from Subject00_1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 90999  =      0.000 ...   181.998 secs...


Subject00_1.edf: 90546 valid rows
Extracting EDF parameters from Subject00_2.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 30999  =      0.000 ...    61.998 secs...
Subject00_2.edf: 30515 valid rows
Extracting EDF parameters from Subject01_1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 90999  =      0.000 ...   181.998 secs...
Subject01_1.edf: 90546 valid rows
Extracting EDF parameters from Subject01_2.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 30999  =      0.000 ...    61.998 secs...
Subject01_2.edf: 30515 valid rows
Extracting EDF parameters from Subject02_1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 90999  =      0.000 ...   181.998 secs...
Subject02_1.edf: 90546 valid rows
Extracting EDF parameters from Subject02_2.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 30999  =      0.0

,time,EEG Fp1,EEG Fp2,EEG F3,EEG F4,EEG F7,EEG F8,EEG T3,EEG T4,EEG C3,...,EEG P3,EEG P4,EEG O1,EEG O2,EEG Fz,EEG Cz,EEG Pz,EEG A2-A1,subject_id,session
0,0.000,-3.647938,-3.581866,-4.081247,-0.237508,-3.460168,-1.021940,-1.619108,4.479832,-1.957309,...,1.076818,7.086144,4.313824,10.168745,-0.227751,0.358397,2.822136,1.291542,0,baseline
1,0.002,-4.236482,-4.279388,-4.766219,-0.617699,-3.709403,-1.308793,-0.877716,5.607640,-1.426855,...,2.962960,9.159618,6.237584,12.274858,-0.602733,4.160351,5.311727,0.973219,0,baseline
2,0.004,-4.954218,-5.020849,-5.783455,-1.193804,-3.903253,-1.523457,0.171455,7.056246,-0.864033,...,5.318983,11.767794,8.766892,14.835672,-1.163293,8.861065,8.367278,0.579025,0,baseline
3,0.006,-5.703854,-5.656125,-7.079280,-1.928968,-4.019562,-1.548153,1.453587,8.670264,-0.369542,...,7.870239,14.621829,11.722487,17.598852,-1.882648,13.699181,11.597705,0.182633,0,baseline
4,0.008,-6.370552,-6.046078,-8.518234,-2.747541,-4.076794,-1.278398,2.842846,10.249195,-0.027893,...,10.264316,17.350356,14.857342,20.304570,-2.682355,17.717742,14.534023,-0.119747,0,baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720362,61.020,-6.225414,11.253639,-1.699319,-7.326075,-1.923019,-3.876462,-5.228226,-17.340635,-2.889137,...,2.558638,14.437941,13.393328,9.322056,-3.436279,-1.041213,9.576340,1.239656,5,task
720363,61.022,-6.225414,11.253639,-1.699319,-7.326075,-1.923019,-3.876462,-5.228226,-17.340635,-2.889137,...,2.558638,14.437941,13.393328,9.322056,-3.436279,-1.041213,9.576340,1.239656,5,task
720364,61.024,-6.225414,11.253639,-1.699319,-7.326075,-1.923019,-3.876462,-5.228226,-17.340635,-2.889137,...,2.558638,14.437941,13.393328,9.322056,-3.436279,-1.041213,9.576340,1.239656,5,task
720365,61.026,-6.225414,11.253639,-1.699319,-7.326075,-1.923019,-3.876462,-5.228226,-17.340635,-2.889137,...,2.558638,14.437941,13.393328,9.322056,-3.436279,-1.041213,9.576340,1.239656,5,task


In [ ]:
#make full second sized windows for proper frequency resolution
fs=500
window_size=500
n_windows=len(df)//window_size

#get all channels as long as they are eeg channels
channels = []
for col in df.columns:
    if col.startswith('EEG'):
        channels.append(col)

print(f"n_windows: {n_windows}")
print(f"expected coherence rows: {n_windows*190}")
print(f"channels: {channels}")

n_windows: 1440
expected coherence rows: 273600
channels: ['EEG Fp1', 'EEG Fp2', 'EEG F3', 'EEG F4', 'EEG F7', 'EEG F8', 'EEG T3', 'EEG T4', 'EEG C3', 'EEG C4', 'EEG T5', 'EEG T6', 'EEG P3', 'EEG P4', 'EEG O1', 'EEG O2', 'EEG Fz', 'EEG Cz', 'EEG Pz', 'EEG A2-A1']


In [ ]:
# band definitions for coherence
# delta, theta, alpha, beta have strong signal
BANDS = {
    'delta':(0.5,4),
    'theta':(4,8),
    'alpha':(8,13),
    'beta':(13,30),
    'gamma':(30,45)
}

# window loop to find band-specific coherence of pairs per window
# no pre-filtering needed: scipy.signal.coherence already returns per-frequency values
# we just mask the output by band range (same technique as band power)
output_list = []
for w in range(n_windows):
    print(f"window #: {w} of {n_windows}")
    start=w*window_size
    end=start+window_size
    window_data=df.iloc[start:end]
    mid = start + window_size // 2
    subj = df.iloc[mid]['subject_id']
    sess = df.iloc[mid]['session']

    for i in range(len(channels)):
        for j in range(i+1, len(channels)):
            channel1 = channels[i]
            channel2 = channels[j]
            freqs, coh = coherence(window_data[channel1].values, window_data[channel2].values, fs=500.0, nperseg=256)
            row = {'epoch_id': w, 'subject_id': subj, 'session': sess, 'channel_i': channel1, 'channel_j': channel2}
            for band, (lo, hi) in BANDS.items():
                mask = (freqs >= lo) & (freqs < hi)
                row[band + '_coherence'] = coh[mask].mean()
            output_list.append(row)

output_df = pd.DataFrame(output_list)

# clean up bad values from dying signal at file boundaries
output_df = output_df.dropna()
output_df = output_df[output_df['delta_coherence'] < 1.0]
output_df = output_df[output_df['theta_coherence'] < 1.0]
output_df = output_df[output_df['alpha_coherence'] < 1.0]
output_df = output_df[output_df['beta_coherence'] < 1.0]
print(f"Total coherence rows: {len(output_df)}")
output_df

window #: 0 of 1440
window #: 1 of 1440
window #: 2 of 1440
window #: 3 of 1440
window #: 4 of 1440
window #: 5 of 1440
window #: 6 of 1440
window #: 7 of 1440
window #: 8 of 1440
window #: 9 of 1440
window #: 10 of 1440
window #: 11 of 1440
window #: 12 of 1440
window #: 13 of 1440
window #: 14 of 1440
window #: 15 of 1440
window #: 16 of 1440
window #: 17 of 1440
window #: 18 of 1440
window #: 19 of 1440
window #: 20 of 1440
window #: 21 of 1440
window #: 22 of 1440
window #: 23 of 1440
window #: 24 of 1440
window #: 25 of 1440
window #: 26 of 1440
window #: 27 of 1440
window #: 28 of 1440
window #: 29 of 1440
window #: 30 of 1440
window #: 31 of 1440
window #: 32 of 1440
window #: 33 of 1440
window #: 34 of 1440
window #: 35 of 1440
window #: 36 of 1440
window #: 37 of 1440
window #: 38 of 1440
window #: 39 of 1440
window #: 40 of 1440
window #: 41 of 1440
window #: 42 of 1440
window #: 43 of 1440
window #: 44 of 1440
window #: 45 of 1440
window #: 46 of 1440
window #: 47 of 1440
wi

,epoch_id,subject_id,session,channel_i,channel_j,delta_coherence,theta_coherence,alpha_coherence,beta_coherence,gamma_coherence
0,0,0,baseline,EEG Fp1,EEG Fp2,0.971521,0.974620,0.887708,0.810394,0.913461
1,0,0,baseline,EEG Fp1,EEG F3,0.813916,0.949081,0.961968,0.650274,0.666310
2,0,0,baseline,EEG Fp1,EEG F4,0.683999,0.670868,0.800318,0.692328,0.831752
3,0,0,baseline,EEG Fp1,EEG F7,0.905256,0.906756,0.955570,0.527214,0.423744
4,0,0,baseline,EEG Fp1,EEG F8,0.493458,0.710236,0.529307,0.558200,0.828505
...,...,...,...,...,...,...,...,...,...,...
273595,1439,5,task,EEG Fz,EEG Pz,0.778895,0.701681,0.913111,0.594996,0.803256
273596,1439,5,task,EEG Fz,EEG A2-A1,0.791857,0.479635,0.520106,0.349008,0.458079
273597,1439,5,task,EEG Cz,EEG Pz,0.571413,0.765323,0.451040,0.620141,0.822958
273598,1439,5,task,EEG Cz,EEG A2-A1,0.332221,0.664279,0.210600,0.327074,0.494949


In [19]:
output_df.to_csv('TDG_trial_run_data.csv')

In [ ]:
# band power per channel per window
# delta, theta, alpha, beta have strong signal
# gamma has minimal signal (data is low-pass filtered at 30Hz) but included for completeness
BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
    'gamma': (30, 45)
}

band_list = []
for w in range(n_windows):
    print(f"Processing band power window {w} of {n_windows}")
    start=w*window_size
    end=start+window_size
    window_data=df.iloc[start:end]
    mid = start + window_size // 2
    subj = df.iloc[mid]['subject_id']
    sess = df.iloc[mid]['session']
    
    for channel in channels:
        segment=window_data[channel].values
        freqs, psd=welch(segment, fs=500.0, nperseg=256)
        
        powers = {}
        for band, (lo, hi) in BANDS.items():
            mask = (freqs >= lo) & (freqs < hi)
            powers[band] = np.sum(psd[mask])
        
        total = sum(powers.values())
        for band in powers:
            powers[band] = powers[band] / total
        
        dominant = max(powers, key=powers.get)
        
        row = {
            'epoch_id': w,
            'subject_id': subj,
            'session': sess,
            'channel': channel,
            'delta': powers['delta'],
            'theta': powers['theta'],
            'alpha': powers['alpha'],
            'beta': powers['beta'],
            'gamma': powers['gamma'],
            'dominant_band': dominant
        }
        band_list.append(row)

band_df = pd.DataFrame(band_list)

# clean up bad values from dying signal at file boundaries
band_df = band_df.dropna()
print(f"Total band power rows: {len(band_df)}")
band_df

Processing band power window 0 of 1440
Processing band power window 1 of 1440
Processing band power window 2 of 1440
Processing band power window 3 of 1440
Processing band power window 4 of 1440
Processing band power window 5 of 1440
Processing band power window 6 of 1440
Processing band power window 7 of 1440
Processing band power window 8 of 1440
Processing band power window 9 of 1440
Processing band power window 10 of 1440
Processing band power window 11 of 1440
Processing band power window 12 of 1440
Processing band power window 13 of 1440
Processing band power window 14 of 1440
Processing band power window 15 of 1440
Processing band power window 16 of 1440
Processing band power window 17 of 1440
Processing band power window 18 of 1440
Processing band power window 19 of 1440
Processing band power window 20 of 1440
Processing band power window 21 of 1440
Processing band power window 22 of 1440
Processing band power window 23 of 1440
Processing band power window 24 of 1440
Processing

,epoch_id,subject_id,session,channel,delta,theta,alpha,beta,gamma,dominant_band
0,0,0,baseline,EEG Fp1,0.495404,0.206765,0.106919,0.149537,0.041375,delta
1,0,0,baseline,EEG Fp2,0.473324,0.217361,0.074882,0.181188,0.053245,delta
2,0,0,baseline,EEG F3,0.505253,0.167638,0.105890,0.185091,0.036128,delta
3,0,0,baseline,EEG F4,0.577215,0.103778,0.044716,0.228933,0.045358,delta
4,0,0,baseline,EEG F7,0.557690,0.162592,0.094005,0.157093,0.028620,delta
...,...,...,...,...,...,...,...,...,...,...
28795,1439,5,task,EEG O2,0.042044,0.100372,0.838641,0.014948,0.003996,alpha
28796,1439,5,task,EEG Fz,0.191824,0.182647,0.379952,0.216824,0.028753,alpha
28797,1439,5,task,EEG Cz,0.178548,0.124164,0.237660,0.424500,0.035127,beta
28798,1439,5,task,EEG Pz,0.240964,0.244673,0.437243,0.060588,0.016531,alpha


In [21]:
band_df.to_csv('TDG_band_power_data.csv')